In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
!pip install ucimlrepo

In [3]:
from ucimlrepo import fetch_ucirepo 

covertype = fetch_ucirepo(id=31) # fetch dataset

x = covertype.data.features #features
y = covertype.data.targets  #class

In [4]:
x.head()

,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,...,Soil_Type34,Soil_Type35,Soil_Type36,Soil_Type37,Soil_Type38,Soil_Type39,Soil_Type40,Wilderness_Area2,Wilderness_Area3,Wilderness_Area4
0,2596,51,3,258,0,510,221,232,148,6279,...,0,0,0,0,0,0,0,0,0,0
1,2590,56,2,212,-6,390,220,235,151,6225,...,0,0,0,0,0,0,0,0,0,0
2,2804,139,9,268,65,3180,234,238,135,6121,...,0,0,0,0,0,0,0,0,0,0
3,2785,155,18,242,118,3090,238,238,122,6211,...,0,0,0,0,0,0,0,0,0,0
4,2595,45,2,153,-1,391,220,234,150,6172,...,0,0,0,0,0,0,0,0,0,0


In [5]:
x.shape

(581012, 54)

In [6]:
y.head()

,Cover_Type
0,5
1,5
2,2
3,2
4,5


In [7]:
y.shape

(581012, 1)

In [8]:
x.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 581012 entries, 0 to 581011
Data columns (total 54 columns):
 #   Column                              Non-Null Count   Dtype
---  ------                              --------------   -----
 0   Elevation                           581012 non-null  int64
 1   Aspect                              581012 non-null  int64
 2   Slope                               581012 non-null  int64
 3   Horizontal_Distance_To_Hydrology    581012 non-null  int64
 4   Vertical_Distance_To_Hydrology      581012 non-null  int64
 5   Horizontal_Distance_To_Roadways     581012 non-null  int64
 6   Hillshade_9am                       581012 non-null  int64
 7   Hillshade_Noon                      581012 non-null  int64
 8   Hillshade_3pm                       581012 non-null  int64
 9   Horizontal_Distance_To_Fire_Points  581012 non-null  int64
 10  Wilderness_Area1                    581012 non-null  int64
 11  Soil_Type1                          581012 non-null 

In [9]:
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report

from sklearn.ensemble import RandomForestClassifier

import warnings as wr
wr.filterwarnings('ignore')

# Train Test Split

In [10]:
xtrain, xtest, ytrain, ytest = train_test_split(x, y, random_state=42, test_size=.30)

In [11]:
print(xtrain.shape,ytrain.shape)
print(xtest.shape,ytest.shape)

(406708, 54) (406708, 1)
(174304, 54) (174304, 1)


# Test and validation

In [12]:
x_test, x_val, y_test, y_val = train_test_split(xtest, ytest, random_state=42, test_size=.50)

In [13]:
print(x_test.shape,y_test.shape)
print(x_val.shape,y_val.shape)

(87152, 54) (87152, 1)
(87152, 54) (87152, 1)


# Random Forest model

In [14]:
rf_model = RandomForestClassifier(random_state=42,n_jobs=-1,verbose=1)

In [15]:
rf_model.fit(xtrain,ytrain)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:   16.1s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   37.5s finished


RandomForestClassifier(n_jobs=-1, random_state=42, verbose=1)

In [16]:
y_predict_rf = rf_model.predict(x_test)

[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.5s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    1.2s finished


In [17]:
y_predict_rf

array([2, 2, 1, ..., 6, 2, 1])

In [18]:
len(y_predict_rf)

87152

# Evaluate Random Forest model

In [19]:
print("=========>Random Forest Model Performance<=============") 
print("Accuracy:", accuracy_score(y_test, y_predict_rf),) 
print("----------Classification Report-------------") 
print(classification_report(y_test, y_predict_rf)) 

=========>Random Forest Model Performance<=============
Accuracy: 0.9515559023315586
----------Classification Report-------------
              precision    recall  f1-score   support

           1       0.96      0.94      0.95     31897
           2       0.95      0.97      0.96     42414
           3       0.94      0.96      0.95      5341
           4       0.94      0.86      0.90       401
           5       0.94      0.75      0.83      1448
           6       0.94      0.89      0.92      2636
           7       0.97      0.96      0.96      3015

    accuracy                           0.95     87152
   macro avg       0.95      0.90      0.92     87152
weighted avg       0.95      0.95      0.95     87152



# Pre-pruning for cost complexity

In [20]:
rf_params = {
    'n_estimators': [150,200],
    'criterion': ['gini', 'entropy','log_loss'],
    'max_depth': [None],
    'min_samples_split': [2],
    'min_samples_leaf': [1],
    'max_features': ['sqrt', 'log2',None]
}

# perform Grid search for Random Forest

In [21]:
rf_grid_search = GridSearchCV(estimator=RandomForestClassifier(random_state=42),param_grid=rf_params,
                             cv=3,n_jobs=-1,verbose=1)

In [23]:
type(ytrain)

pandas.core.frame.DataFrame

In [27]:
ytrain_reshaped = ytrain.iloc[:, 0].to_numpy()

In [29]:
type(ytrain_reshaped)

numpy.ndarray

In [30]:
ytrain_reshaped.shape

(406708,)

In [32]:
rf_grid_search.fit(xtrain,ytrain_reshaped)

Fitting 3 folds for each of 18 candidates, totalling 54 fits


GridSearchCV(cv=3, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy', 'log_loss'],
                         'max_depth': [None],
                         'max_features': ['sqrt', 'log2', None],
                         'min_samples_leaf': [1], 'min_samples_split': [2],
                         'n_estimators': [150, 200]},
             verbose=1)

In [33]:
pd.DataFrame(rf_grid_search.cv_results_)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_depth,param_max_features,param_min_samples_leaf,param_min_samples_split,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score
0,149.464448,0.213657,11.773683,0.267214,gini,None,sqrt,1,2,150,"{'criterion': 'gini', 'max_depth': None, 'max_...",0.942605,0.942870,0.943837,0.943104,0.000529,12
1,194.516757,2.547473,14.028570,0.483765,gini,None,sqrt,1,2,200,"{'criterion': 'gini', 'max_depth': None, 'max_...",0.943092,0.943188,0.944294,0.943525,0.000545,11
2,122.003727,1.085808,11.366607,0.937104,gini,None,log2,1,2,150,"{'criterion': 'gini', 'max_depth': None, 'max_...",0.936948,0.937419,0.938297,0.937555,0.000559,17
3,165.114531,1.193324,14.954278,0.926108,gini,None,log2,1,2,200,"{'criterion': 'gini', 'max_depth': None, 'max_...",0.936749,0.937655,0.938238,0.937547,0.000613,18
4,652.913310,2.174763,7.719359,0.171004,gini,None,None,1,2,150,"{'criterion': 'gini', 'max_depth': None, 'max_...",0.958420,0.957638,0.957070,0.957709,0.000554,6
5,861.568284,7.430003,10.322980,0.205687,gini,None,None,1,2,200,"{'criterion': 'gini', 'max_depth': None, 'max_...",0.958641,0.957852,0.957387,0.957960,0.000518,5
6,143.830073,1.896211,9.470548,0.621787,entropy,None,sqrt,1,2,150,"{'criterion': 'entropy', 'max_depth': None, 'm...",0.945593,0.945651,0.946109,0.945784,0.000231,9
7,194.363304,3.254707,12.257251,0.203691,entropy,None,sqrt,1,2,200,"{'criterion': 'entropy', 'max_depth': None, 'm...",0.945696,0.946094,0.946544,0.946111,0.000346,7
8,124.522715,4.034483,9.781089,0.316767,entropy,None,log2,1,2,150,"{'criterion': 'entropy', 'max_depth': None, 'm...",0.939279,0.939595,0.940016,0.939630,0.000302,15
9,167.339378,1.701823,13.564344,0.798014,entropy,None,log2,1,2,200,"{'criterion': 'entropy', 'max_depth': None, 'm...",0.939581,0.940016,0.940215,0.939937,0.000265,13


In [34]:
rf_grid_search.best_params_

{'criterion': 'entropy',
 'max_depth': None,
 'max_features': None,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'n_estimators': 200}

In [35]:
rf_grid_search_best_model = rf_grid_search.best_estimator_

In [36]:
rf_grid_y_predict = rf_grid_search_best_model.predict(x_test)

In [38]:
rf_grid_y_predict

array([2, 2, 1, ..., 6, 2, 1])

In [41]:
print("----------Accuracy & Classification Report-----------")
print(classification_report(y_test,rf_grid_y_predict))

----------Accuracy & Classification Report-----------
              precision    recall  f1-score   support

           1       0.97      0.97      0.97     31897
           2       0.97      0.98      0.97     42414
           3       0.96      0.97      0.97      5341
           4       0.93      0.90      0.92       401
           5       0.93      0.86      0.90      1448
           6       0.95      0.94      0.94      2636
           7       0.98      0.97      0.97      3015

    accuracy                           0.97     87152
   macro avg       0.96      0.94      0.95     87152
weighted avg       0.97      0.97      0.97     87152



In [42]:
rf_grid_search_best_model

RandomForestClassifier(criterion='entropy', max_features=None, n_estimators=200,
                       random_state=42)

In [44]:
rf_grid_search_best_model.score(x_test,y_test)

0.9685606756012484